# Flow Chart Pengerjaan:
1. Cover Image
2. DCT & Quantization
3. Block Smoothness Estimation & Sorting
4. Zigzag Scan
5. NACP Construction
6. Adaptive Hexagonal Payload Assignment
7. Hexagonal Turtle Shell Embedding
8. Stego DCT Coefficients
9. Entropy Coding
10. Stego Image

In [51]:
!pip install jpeglib numpy matplotlib opencv-python-headless scipy scikit-image seaborn pandas tqdm import-ipynb 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [52]:
from PIL import Image
from performance import psnr, fsi, ssim
from zigzag import zigzag, inverse_zigzag
from math import ceil, floor, log2, log10, sqrt
import copy
import cv2
import jpeglib
import numpy as np
import import_ipynb
import matplotlib.pyplot as plt
import turtleShell
import FrequencyDomain as FD

In [53]:
def change_image_QF_high_quality(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)

    dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(target_qf)
    im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
    im.qt[0] = FD.custom_q_mat(100)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}_hd.jpeg"
    im.write_dct(output_path)

In [54]:
def change_image_QF(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]

    # From QF 100 to target QF
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    
    # Update image object
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    
    ori_path =  image_path.split(".jpeg")[0]
    output_path = f"{ori_path}_qf{target_qf}.jpeg"
    im.write_dct(output_path)

In [55]:
def get_quantized_coefficients(image_path):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, _, _  = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [56]:
def get_compress_coeff(image_path, target_qf):
    im = jpeglib.read_dct(image_path)
    old_qt = im.qt[0]
    dequantized = im.Y.astype(np.float64) * old_qt
    new_coefficients = np.round(dequantized / FD.custom_q_mat(target_qf)).astype(np.int16)
    im.Y[:] = new_coefficients
    im.qt[0] = FD.custom_q_mat(target_qf)
    num_v_blocks, num_h_blocks, _, _ = im.Y.shape
    sorted_coeffs = []
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block = im.Y[i, j]
            zigzag_coeffs = zigzag(block)
            sorted_coeffs.append(zigzag_coeffs)
    return sorted_coeffs

In [57]:
def convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100):
    with Image.open(tiff_path) as img:
        # rgb_img = img.convert('RGB')  # Convert to RGB if it's not already
        # rgb_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
        gray_img = img.convert('L')  # Convert to grayscale
        gray_img.save(jpeg_path, 'JPEG', quality=quality, subsampling=0, optimize=False)
    print(f"Converted {tiff_path} to {jpeg_path} with quality {quality}")

In [58]:
def sort_smoothness(smoothness_list):
    smoothness_list.sort(key=lambda x: (-x[1], x[2]))
    return smoothness_list

In [59]:
def block_smoothness_estimation(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    smoothness_block = []
    total_ec = 0
    total_zero_count = 0
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            print( block)
            block_1d = block.flatten()
            block_1d = block_1d[1:]  # AC coefficients 
            zero_count = np.sum(block_1d == 0)
            non_zero_sum = np.sum(abs(block_1d[block_1d != 0]))
            non_zero_indices = np.nonzero(block_1d)[0]
            capable_bits = 4 if zero_count == 0 else 3
            total_ec += len(non_zero_indices) // 2 * capable_bits
            total_zero_count += zero_count
            smoothness_block.append(((i, j), zero_count, non_zero_sum))

    print(f"Total blocks: {h * w}")
    print(f"Total embedding capacity (estimated): {total_ec} bits")
    print(f"Total zero count: {total_zero_count}")
    # return smoothness_block

def block_smoothness(image):
    im = jpeglib.read_dct(image)
    h, w, _, _ = im.Y.shape
    q_table = im.qt[0]
    smoothness_block = []
    smoothness_score = []
    for i in range(h):
        for j in range(w):
            block = im.Y[i, j]
            ac_block = block.copy()
            ac_block[0, 0] = 0
            z_k = np.sum(ac_block == 0)
            E_k = np.sum((ac_block != 0) * (q_table ** 2))
            S_k = z_k + float(z_k / E_k)
            smoothness_block.append(((i, j), z_k, E_k, S_k))
            smoothness_score.append(((i, j), S_k))
    
    print(f"Total blocks: {h * w}")
    print(smoothness_block)
    print(smoothness_score)
    return smoothness_block, smoothness_score

In [60]:
def invariant_ac_smoothness(image_path, threshold_eob):
    coeffs = get_quantized_coefficients(image_path)
    smoothness_block = []
    for idx in range(len(coeffs)):
        sum_z_k = 0
        sum_ac_k = 0
        for k in range(threshold_eob + 1, 64):
            if coeffs[idx][k] == 0:
                sum_z_k += 1
            sum_ac_k += abs(coeffs[idx][k])
        smoothness_block.append(((idx), sum_z_k, sum_ac_k))
    s_smoothness_block = sort_smoothness(smoothness_block)
    return smoothness_block, s_smoothness_block

In [61]:
def get_nacp(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

def get_nacp_2(sorted_coefficients):
    valid_nacp = []
    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:] # AC coefficients
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) > 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for i in range(0, len(non_zero_ac) - 1, 2):
            x = int(non_zero_ac[i])
            y = int(non_zero_ac[i+1])
            if x != 0 and y != 0:
                valid_nacp.append((x, y))

    return valid_nacp

In [62]:
def replace_nacp(sorted_coefficients, nacp_coords):
    pair_index = 0

    for zigzag_coeff in sorted_coefficients:
        ac_coeffs = zigzag_coeff[1:]  
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        non_zero_ac = [ac_coeffs[i] for i in non_zero_indices]

        for idx in range(0, len(non_zero_ac) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]
                i1, i2 = non_zero_indices[idx], non_zero_indices[idx + 1]
                ac_coeffs[i1] = float(new_x)
                ac_coeffs[i2] = float(new_y)
                pair_index += 1
            else: break
        zigzag_coeff[1:] = ac_coeffs

    return sorted_coefficients

def replace_nacp_2(sorted_coefficients, nacp_coords):
    pair_index = 0
    for zigzag_coeff in sorted_coefficients:
        ac_part = zigzag_coeff[1:]
        rel_non_zero_indices = np.nonzero(np.abs(ac_part) > 1)[0]        
        for idx in range(0, len(rel_non_zero_indices) - 1, 2):
            if pair_index < len(nacp_coords):
                new_x, new_y = nacp_coords[pair_index]                
                zigzag_coeff[rel_non_zero_indices[idx] + 1] = float(new_x)
                zigzag_coeff[rel_non_zero_indices[idx+1] + 1] = float(new_y)
                pair_index += 1
            else: break
    return sorted_coefficients

In [63]:
def construct_stego_file(image_path, new_coeffs):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

def construct_stego_file_2(image_path, new_coeffs, qf):
    im = jpeglib.read_dct(image_path)
    num_v_blocks, num_h_blocks, v_block_size, h_block_size  = im.Y.shape
    idx = 0
    
    for i in range(num_v_blocks):
        for j in range(num_h_blocks):
            block_coeffs = new_coeffs[idx]
            block = inverse_zigzag(block_coeffs, v_block_size, h_block_size)
            im.Y[i, j] = block
            idx += 1

    if qf is not None:    
        dequantized = im.Y.astype(np.float64) * FD.custom_q_mat(qf)
        im.Y[:] = np.round(dequantized / FD.custom_q_mat(100)).astype(np.int16)
        im.qt[0] = FD.custom_q_mat(100)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Image with secret data is saved to {output_path}")
    im.write_dct(output_path)

In [64]:
def data_hiding_process(secret_data, nacp_coord="", mode="8N"):
    if secret_data == "": return nacp_coord
    bit = 3 if mode == "8N" else 4
    secret_data = secret_data + '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)
    print(f"Secret Data: {secret_data}")
    print(f"Panjang Bit Secret Data: {lendata}")

    decimals = []
    for i in range(0, len(data_bin), bit):
        group = data_bin[i:i+bit].ljust(bit, '0')
        decimals.append(int(group, 2))

    _, shells, cell_to_shells = turtleShell.init(mode) 
    if len(decimals) > len(nacp_coord):
        print("Warning: Not enough NACP coordinates to embed all data.")
        
    for i in range(len(decimals)):
        x, y = nacp_coord[i]
        if turtleShell.get_hex_matrix_value(x, y, mode) == decimals[i]:
            nacp_coord[i] = (x, y)
        else:
            # shell_coords = turtleShell.get_kxk_nearest_signed(x, y, bit)
            _, shell_coords = turtleShell.get_shell_coords(x, y, shells, cell_to_shells)
            found = turtleShell.find_corresponding_val(shell_coords, decimals[i], nacp_coord[i], mode)
            nacp_coord[i] = found
            
    return nacp_coord

In [65]:
def data_extract_process(nacp_coord, mode="8N"):
    extracted_data = ""
    data_bits = ""

    for i, (x, y) in enumerate(nacp_coord):
        val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
        bit = 3 if mode == "8N" else 4
        bits = format(val & ((1 << bit) - 1), f'0{bit}b')
        data_bits += bits

        while len(data_bits) >= 8:
            byte = data_bits[:8]
            char_val = int(byte, 2)
            if char_val == 0: # Null terminator ASCII
                return extracted_data
            try:
                char = chr(char_val)
                extracted_data += char
            except:
                return extracted_data
            data_bits = data_bits[8:]

    return extracted_data

In [66]:
def encode(image_path, message_bits):
    sorted_coeffs = get_quantized_coefficients(image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(message_bits, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file(image_path, modified_coeffs)
    print("Data embedding completed.")

def encode_2(image_path, data, qf):
    image = Image.open(image_path).convert('L')
    stegoimg = image.copy()
    img_arr = np.array(stegoimg)
    q_mat = FD.custom_q_mat(qf)
    sorted_coefficients = FD.transform_to_freq(img_arr, q_mat)
    nacp_coords = get_nacp(sorted_coefficients)
    modified_nacp_coords = data_hiding_process(data, nacp_coords)
    modified_coeffs = replace_nacp(sorted_coefficients, modified_nacp_coords)
    np.save("modified_coefficients.npy", modified_coeffs)

def encode_4(image_path, secret_data, qf):
    sorted_coeffs = get_compress_coeff(image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    print(f"NACP Length: {len(nacp_coords)}")
    print(f"Total EC: {len(nacp_coords * 3)}")
    modified_nacp_coords = data_hiding_process(secret_data, nacp_coords, mode="8N")
    modified_coeffs = replace_nacp(sorted_coeffs, modified_nacp_coords)
    construct_stego_file_2(image_path, modified_coeffs, qf)
    
# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def encode_3(image_path, secret_data):
    _, smoothness_score = block_smoothness(image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    secret_data += '\0'
    data_bin = ''.join(format(ord(c), '08b') for c in secret_data)
    lendata = len(data_bin)

    _, shells, cell_to_shells = turtleShell.init(mode="8N")
    _, shells_17, cell_to_shells_17 = turtleShell.init(mode="17N")
    
    im = jpeglib.read_dct(image_path)
    for (block_i, block_j), score in smoothness_list:
        # print(f"Processing block ({block_i}, {block_j}) with smoothness score {score}")
        if lendata <= 0: break
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]
        choosen_shells = shells if N == 8 else shells_17
        choosen_cell_to_shells = cell_to_shells if N == 8 else cell_to_shells_17
        for idx in range(0, len(non_zero_indices) - 1, 2):
            if lendata <= 0: break
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            bits = data_bin[:t].ljust(t, '0') 
            data_bin = data_bin[t:]
            lendata -= t
            target_val = int(bits, 2)
            val_int = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            if target_val != val_int:
                _, shell_coords = turtleShell.get_shell_coords(x, y, choosen_shells, choosen_cell_to_shells)
                x, y = turtleShell.find_corresponding_val(shell_coords, target_val, (x, y), mode=mode)
            ac_coeffs[non_zero_indices[idx]] = float(x)
            ac_coeffs[non_zero_indices[idx + 1]] = float(y)

        zigzag_coeffs[1:] = ac_coeffs
        zigzag_coeffs[0] = block[0, 0]  
        im.Y[block_i, block_j] = inverse_zigzag(zigzag_coeffs, 8, 8)

    output_path = "stego-images/stego_" + image_path.split("/")[-1]
    print(f"Data embedding completed. Stego image saved to {output_path}")
    im.write_dct(output_path)

In [67]:
def decode(stego_image_path):
    sorted_coeffs = get_quantized_coefficients(stego_image_path)
    nacp_coords = get_nacp(sorted_coeffs)
    print(nacp_coords)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

def decode_2(stego_file):
    modified_coeffs = np.load(stego_file, allow_pickle=True)
    nacp_coords = get_nacp(modified_coeffs)
    extracted_data = data_extract_process(nacp_coords)
    return extracted_data

def decode_4(stego_image_path, qf):
    sorted_coeffs = get_compress_coeff(stego_image_path, qf)
    nacp_coords = get_nacp(sorted_coeffs)
    extracted_data = data_extract_process(nacp_coords, mode="8N")
    return extracted_data

# Proposed Method - Adaptive Payload in Multi Turtle Shell Embedding
def decode_3(stego_image_path):
    _, smoothness_score = block_smoothness(stego_image_path)
    smoothness_list = sorted(smoothness_score, key=lambda x: -x[1])
    im = jpeglib.read_dct(stego_image_path)
    bitstream = ""
    decoded_text = ""
    for (block_i, block_j), score in smoothness_list:
        N = 8 if score <= threshold else 17
        t = 3 if N == 8 else 4
        mode = f"{N}N"
        block = im.Y[block_i, block_j]
        zigzag_coeffs = zigzag(block)
        ac_coeffs = zigzag_coeffs[1:]
        non_zero_indices = np.nonzero(np.abs(ac_coeffs) >= 1)[0]

        for idx in range(0, len(non_zero_indices) - 1, 2):
            x = int(ac_coeffs[non_zero_indices[idx]])
            y = int(ac_coeffs[non_zero_indices[idx + 1]])
            val = turtleShell.get_hex_matrix_value(x, y, mode=mode)
            bits = format(val, f"0{t}b")
            bitstream += bits
            while len(bitstream) >= 8:
                byte = bitstream[:8]
                bitstream = bitstream[8:]
                char_val = int(byte, 2)
                if char_val == 0:   # Null terminator
                    return decoded_text
                decoded_text += chr(char_val)
    return decoded_text

In [68]:
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()
    return content

In [69]:
cover_images = [
    "cover-images/misc/boat.512.tiff",
    "cover-images/misc/4.2.01.tiff",
    "cover-images/misc/4.2.03.tiff",
    "cover-images/misc/4.2.05.tiff",
    "cover-images/misc/4.2.06.tiff",
    "cover-images/misc/4.2.07.tiff"
]

jpeg_images_path = [
    "cover-images/boat.jpeg",
    "cover-images/splash.jpeg",
    "cover-images/baboon.jpeg",
    "cover-images/airplane.jpeg",
    "cover-images/lake.jpeg",
    "cover-images/peppers.jpeg"
]

# convert_tiff_to_jpeg("cover-images/misc/boat.512.tiff", "cover-images/boat_qf100.jpeg", quality=100)

for tiff_path, jpeg_path in zip(cover_images, jpeg_images_path):
    convert_tiff_to_jpeg(tiff_path, jpeg_path, quality=100)

Converted cover-images/misc/boat.512.tiff to cover-images/boat.jpeg with quality 100
Converted cover-images/misc/4.2.01.tiff to cover-images/splash.jpeg with quality 100
Converted cover-images/misc/4.2.03.tiff to cover-images/baboon.jpeg with quality 100
Converted cover-images/misc/4.2.05.tiff to cover-images/airplane.jpeg with quality 100
Converted cover-images/misc/4.2.06.tiff to cover-images/lake.jpeg with quality 100
Converted cover-images/misc/4.2.07.tiff to cover-images/peppers.jpeg with quality 100


In [70]:
pay_size = 3 * 1000
cover_folder = "cover-images/"
stego_folder = "stego-images/"
payload_folder = "payload/"
cover_image_path = f"baboon_qf90_hd.jpeg"
stego_image_path = f"stego_{cover_image_path}"
data = read_text_file(f"{payload_folder}{pay_size}bits.txt")
# encode_4(f"{cover_folder}{cover_image_path}", data, 50)
encode(f"{cover_folder}{cover_image_path}", data)

secret_data = decode(f"{stego_folder}{stego_image_path}")
# secret_data = decode_4(f"{stego_folder}{stego_image_path}", 50)
print("Extracted Data:", secret_data) 

[(82, -22), (3, 32), (-76, 33), (78, -33), (21, 28), (27, -57), (96, -35), (-24, 50), (15, 8), (10, -10), (-14, -14), (6, 80), (-24, 40), (-72, 55), (22, -22), (-18, 13), (34, 33), (-22, 48), (17, 36), (-15, 23), (-18, 20), (72, -16), (123, -30), (-32, 12), (-6, -72), (45, 28), (-42, 3), (-32, -15), (-72, 125), (-10, -72), (4, -5), (20, -21), (-7, -6), (24, -10), (12, 11), (-22, -11), (13, 14), (18, 13), (-28, -17), (-14, 64), (16, 17), (-21, -21), (-12, 15), (-22, -20), (-18, -20), (108, 52), (90, 84), (44, -36), (66, -24), (-9, 24), (15, 21), (40, -50), (-32, 10), (60, -4), (-16, -15), (10, -7), (-7, 6), (40, 24), (10, 24), (33, -11), (-26, 28), (36, 26), (-14, 34), (-14, -11), (16, -19), (42, -12), (-46, 24), (80, 44), (-24, -20), (-152, -183), (162, -60), (-45, -48), (-42, 66), (-28, -5), (-135, 110), (-60, 12), (-20, -7), (-21, 78), (-48, 24), (-20, -12), (11, -30), (11, -44), (-16, 39), (-14, 17), (14, -11), (16, -16), (-17, -19), (21, -21), (12, 15), (-23, 24), (22, -20), (24, -

In [71]:
# Test performance metrics
psnr_value = psnr(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
fsi_value = fsi(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
ssim_value = ssim(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print(f"PSNR: {psnr_value} dB")
print(f"FSI: {fsi_value}")
print(f"SSIM: {ssim_value}")

Size cover: 226275
Size stego: 226289
PSNR: 69.63276368868507 dB
FSI: 14.0
SSIM: 0.9999988648520511


In [72]:
def compare_spatial_frequency(cover_image_path, stego_image_path):
    cover = np.array(Image.open(cover_image_path).convert('L'), dtype=np.float64)
    stego = np.array(Image.open(stego_image_path).convert('L'), dtype=np.float64)

    spatial_difference = np.abs(cover - stego) ** 2

    cover_freq = np.fft.fft2(cover)
    stego_freq = np.fft.fft2(stego)
    freq_difference = np.abs(cover_freq - stego_freq)

    return freq_difference, spatial_difference

In [73]:
freq_diff, spatial_diff = compare_spatial_frequency(f"{cover_folder}{cover_image_path}", f"{stego_folder}{stego_image_path}")
print("Frequency Difference:")
for row in freq_diff:
    print(row)
print("Spatial Difference:")
for row in spatial_diff:
    print(row)
print("Max spatial diff:", spatial_diff.max())
print("Mean spatial diff:", spatial_diff.mean())

Frequency Difference:
[ 33.           8.53349612  52.22130894  20.49466349  30.40935736
  32.08319382   7.69584215  14.71649665  22.97610519  22.82095907
   9.75794949  10.92151297   3.50049455  23.20951802  21.25230489
  36.99726001  31.63259682  28.47998916  16.5259603    5.10631596
  59.23781583  32.64098577  30.73545929   7.77093473  35.67579378
  12.03005668  24.88775551   3.90215695  17.76205752   5.7855546
  60.83667136  79.18306735  12.52951614  44.22638395   8.11850437
  40.24465972  21.36548449  28.36220731   8.72020446  74.12304339
  39.96789318  30.2932339   13.50358887  53.48995745   5.90947329
  19.31942339  68.15796174 112.55712564  42.0295953   39.35388499
  22.8849893   57.54272574  24.1958045   40.57832759  20.37436824
  33.39350668  33.18684173  40.19059492  61.19316291  20.11238799
  36.13554473  12.64259853  14.08634678  44.70583953  27.39856275
  25.22759326  26.55696704  38.11633376  13.47074373  14.83982216
  43.5564945   32.88619881  25.77013626  58.28853885  5

In [74]:
max_pixel = 255.0
psnr_value = 20 * log10(max_pixel / sqrt(spatial_diff.mean()))
print("PSNR calculated from spatial difference:", psnr_value, "dB")

PSNR calculated from spatial difference: 69.63276368868507 dB


In [75]:
image_base = [
    "cover-images/baboon.jpeg",
    "cover-images/airplane.jpeg",
    "cover-images/lake.jpeg",
    "cover-images/peppers.jpeg",
    "cover-images/splash.jpeg",
    "cover-images/boat.jpeg"
]

qf = [50, 60, 70, 80, 90]

for img in image_base:
    for q in qf:
        change_image_QF(img, q)
        change_image_QF_high_quality(img, q)